# Financial Modeling Prep ELT Pipeline Documentation

This is a documentation of the ETL Pipeline set up to extract financial data from [FMP](https://site.financialmodelingprep.com/). The extracted data are associated with these five companies: "AAPL", "MSFT", "TSLA", "NVDA", and "AMZN".

## Architecture:
1. Extract from [FMP API](https://site.financialmodelingprep.com/developer/docs)
2. Load SQLite3 into a DB file for staging
3. Transform data
4. Load from DB file to google sheets


## Process Documentation

### I. Extract

The data will be extracted through [FMP API](https://site.financialmodelingprep.com/developer/docs).

#### A. Table Name and Endpoints:

1. *company_profile* - https://financialmodelingprep.com/stable/profile
2. *company_ratios* - https://financialmodelingprep.com/stable/ratios
3. *company_income* - https://financialmodelingprep.com/stable/income-statement
4. *company_quote* - https://financialmodelingprep.com/stable/quote
5. *company_key_metrics* - https://financialmodelingprep.com/stable/key-metrics

#### B. Script

##### 1. Preparations and Declarations

In this step, python libraries to be used are imported, local paths to be accessed are declared, specific companies and endpoints are specified. The existence of a database (nyse_financials) is also assured.

Two options for retrieving the API key:
1. If ran locally, API key will be retrieved from keys.json
2. Authorized through Github actions through GITHUB_TOKEN

In [ ]:
# 1. IMPORT LIBRARIES
import requests       # For API calls
from pathlib import Path  # For local file paths
import json           # For reading credentials
import sqlite3 as sql # For database connection
import pandas as pd   # For dataframes
import time           # For timestamps and pacing API calls
import os

# 2. DEFINE FILE PATHS, VARIABLES, AND API ENDPOINTS
key_path = Path("/Users/addyhalos/Documents/secrets/keys.json")  # Local API key (not stored in repo)
companies = ["AAPL", "MSFT", "TSLA", "NVDA", "AMZN"]             # Example company symbols
data_dir = Path("/Users/addyhalos/Documents/GitHub/datascience-portfolio/data_engineering/data")
db_path = data_dir / "nyse_financials.db"                        # Database file (staging + final)
sleep = 0.5                                                      # Pause between API calls

# API endpoints to extract data from
endpoints = {
    "company_profile": "https://financialmodelingprep.com/stable/profile",
    "company_ratios": "https://financialmodelingprep.com/stable/ratios",
    "company_income": "https://financialmodelingprep.com/stable/income-statement",
    "company_quote": "https://financialmodelingprep.com/stable/quote",
    "company_key_metrics": "https://financialmodelingprep.com/stable/key-metrics"
}


# 3. LOAD API KEY 
# Option 1: Local Key - This is for running this script in your local machine. You will need a local copy of the API Key in the key_path
if not key_path.exists():
    raise FileNotFoundError(f"keys.json not found at {key_path}")
with key_path.open("r", encoding="utf-8") as fh:
    data = json.load(fh)
api_key = data.get("fmp_key")

"""
# Option 2: Github Secret - Use when running on Github Actions - API Key is saved in Github Secrets for this to work
data_dir = Path("data_engineering/data/nyse_financials.db") #change path to github repository path
if not os.getenv("FMP_KEY").exists():
    raise FileNotFoundError(f"FMP_KEY not found in Github Secret")
else:
    api_key = os.getenv("FMP_KEY")
"""

#### 2. Extract, Load, and Transform

1. The presence of nyse_financials.db is checked. If it is non existent, it is created. If it is present, it is loaded in preparation for loading data.

2. The script will loop through 2 lists: the list of companies and the list of endpoints. Each company code will go over all endpoints, and an API call will be made for each. Responses will be recorded in dataframes.

3. fetched_at values added to be used as date reference. Data is also cleaned:
+ Drop duplicate records (same company symbol)
+ Remove failed rows during API extraction
+ Standardize column names
+ Drop unecessary metadata columns
+ Handle missing or invalid values
+ Add timestamp for cleaned data

4. Once all data is extracted, it is loaded into corrseponding tables in nyse_financials.db for staging.

In [ ]:
# 4. CONNECT TO DATABASE (STAGING + FINAL IN SAME FILE)
data_dir.mkdir(exist_ok=True)
session = requests.Session()
conn = sql.connect(db_path)


# 5. EXTRACT AND LOAD INTO STAGING TABLES
try:
    for table_name, endpoint_url in endpoints.items():
        staging_table = f"staging_{table_name}"  # e.g. staging_company_profile
        all_dfs = []
        print(f"\n--- Extracting from endpoint: {table_name} ---")

        # Iterate through each company symbol
        for symbol in companies:
            try:
                resp = session.get(endpoint_url, params={"symbol": symbol, "apikey": api_key}, timeout=20)
                status = resp.status_code
                text_len = len(resp.text or "")
                print(f"{table_name} | {symbol} -> status: {status} | len: {text_len}")

                # Convert API response into DataFrame
                if status == 200:
                    payload = resp.json()
                    if isinstance(payload, list):
                        df = pd.DataFrame(payload) if payload else pd.DataFrame([{"note": "empty_list"}])
                    elif isinstance(payload, dict):
                        df = pd.DataFrame([payload])
                    else:
                        df = pd.DataFrame([{"raw": str(payload)}])
                else:
                    df = pd.DataFrame([{"error_status": status, "error_text": resp.text[:500]}])

                # Annotate metadata
                df["symbol"] = symbol
                df["endpoint"] = table_name
                df["fetched_at"] = pd.Timestamp.utcnow()
                all_dfs.append(df)

            except Exception as e:
                print(f"EXCEPTION: {table_name} | {symbol} -> {e}")
                all_dfs.append(pd.DataFrame([{
                    "symbol": symbol,
                    "endpoint": table_name,
                    "error": str(e),
                    "fetched_at": pd.Timestamp.utcnow()
                }]))
            finally:
                time.sleep(sleep)

        # Combine all rows for that endpoint
        if all_dfs:
            endpoint_df = pd.concat(all_dfs, ignore_index=True, sort=False)
        else:
            endpoint_df = pd.DataFrame([{"note": "no_data_collected", "endpoint": table_name, "fetched_at": pd.Timestamp.utcnow()}])

        # Write to staging table
        endpoint_df.to_sql(staging_table, conn, if_exists="append", index=False)
        print(f"✅ WROTE {len(endpoint_df)} rows to `{staging_table}`")

# 6. TRANSFORM: CLEAN AND LOAD INTO FINAL TABLES
    for table_name in endpoints.keys():
        staging_table = f"staging_{table_name}"
        print(f"\n--- Transforming data from {staging_table} to {table_name} ---")

        # Load from staging
        df = pd.read_sql(f"SELECT * FROM {staging_table}", conn)

# 7. DATA CLEANING

        # 1️⃣ Drop duplicate records (same company symbol)
        df = df.drop_duplicates(subset=["symbol"], keep="last")

        # 2️⃣ Remove any rows that failed during API extraction
        if "error_status" in df.columns:
            df = df[df["error_status"].isna()]
            df = df.drop(columns=["error_status"], errors="ignore")

        # 3️⃣ Standardize column names (lowercase, underscores)
        df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]

        # 4️⃣ Drop unnecessary metadata columns
        df = df.drop(columns=["endpoint"], errors="ignore")

        # 5️⃣ Handle missing or invalid values
        # Example: Replace empty strings with NaN, fill some numeric NaNs with 0
        df = df.replace("", pd.NA)
        num_cols = df.select_dtypes(include=["number"]).columns
        df[num_cols] = df[num_cols].fillna(0)

        # 6️⃣ Add a timestamp for cleaned data
        df["cleaned_at"] = pd.Timestamp.utcnow()

        # ======== LOAD INTO FINAL TABLE ========
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"✅ Loaded cleaned data into `{table_name}` ({len(df)} rows)")

#### 2. Extract, Load, and Transform

In this step, the contents of each table is compared. It will be easy to see if there are discrepancies in the staging and final tables.

In [ ]:
finally:
    # 8. VALIDATION AND SUMMARY
    conn.commit()
    tbls = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
    print("\n✅ DATABASE TABLES PRESENT:")
    for t in tbls['name']:
        cnt = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM `{t}`", conn).iloc[0]['cnt']
        print(f" - {t}: {cnt} rows")
    conn.close()